In [3]:
import torch
import torch.nn as nn
import time
from thop import profile  # 用于精确计算FLOPs

# -------------------------- 配置参数（对齐DeepSeek-V2核心比例） --------------------------
BATCH_SIZE = 2
SEQ_LEN = 2048
HIDDEN_DIM = 4096
NUM_HEADS = 32
HEAD_DIM = 128
LATENT_DIM = 512  # MLA核心：潜空间维度 << NUM_HEADS * HEAD_DIM (4096)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

# -------------------------- 1. 标准多头注意力(MHA) --------------------------
class StandardMHA(nn.Module):
    def __init__(self, hidden_dim, num_heads, head_dim):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.total_kv_dim = num_heads * head_dim
        
        # MHA：3个全维度投影矩阵
        self.w_q = nn.Linear(hidden_dim, self.total_kv_dim, bias=False)
        self.w_k = nn.Linear(hidden_dim, self.total_kv_dim, bias=False)
        self.w_v = nn.Linear(hidden_dim, self.total_kv_dim, bias=False)
        self.w_o = nn.Linear(self.total_kv_dim, hidden_dim, bias=False)
        
    def forward(self, x):
        B, L, _ = x.shape
        
        # 【MHA开销点1】生成并存储全维度Q/K/V (B, L, H*D)
        q = self.w_q(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.w_k(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.w_v(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        
        # 【MHA开销点2】全维度注意力计算 (B, H, L, L)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = scores.softmax(dim=-1)
        output = torch.matmul(attn, v)
        
        output = output.transpose(1, 2).contiguous().view(B, L, self.total_kv_dim)
        return self.w_o(output)

# -------------------------- 2. 多头潜在注意力(MLA) 核心实现 --------------------------
class MultiHeadLatentAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, head_dim, latent_dim):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.latent_dim = latent_dim
        self.total_q_dim = num_heads * head_dim
        
        # MLA：Q保留全维度（保证表达力），KV压缩到低维潜空间
        self.w_q = nn.Linear(hidden_dim, self.total_q_dim, bias=False)
        # 【核心节省1】用1个低维投影代替MHA的2个全维度KV投影
        self.w_c_kv = nn.Linear(hidden_dim, latent_dim, bias=False)
        # 【核心技巧】潜空间到KV的重建权重（不参与中间大矩阵生成）
        self.w_k_latent = nn.Linear(latent_dim, self.total_q_dim, bias=False)
        self.w_v_latent = nn.Linear(latent_dim, self.total_q_dim, bias=False)
        self.w_o = nn.Linear(self.total_q_dim, hidden_dim, bias=False)
        
    def forward(self, x):
        B, L, _ = x.shape

        # Q
        q = self.w_q(x).view(
            B, L, self.num_heads, self.head_dim
        ).transpose(1, 2)  # (B,H,L,D)

        # latent KV
        c_kv = self.w_c_kv(x)  # (B,L,C)

        # ---------------- K projection ----------------

        w_k = self.w_k_latent.weight.view(
            self.num_heads,
            self.head_dim,
            self.latent_dim
        )  # (H,D,C)

        q_k_proj = torch.einsum(
            "bhld,hdc->bhlc",
            q,
            w_k
        )  # (B,H,L,C)

        scores = torch.matmul(
            q_k_proj,
            c_kv.unsqueeze(1).transpose(-2, -1)
        ) / (self.head_dim ** 0.5)

        attn_weights = scores.softmax(dim=-1)

        # ---------------- V aggregation ----------------

        attn_c = torch.matmul(
            attn_weights,
            c_kv.unsqueeze(1)
        )  # (B,H,L,C)

        w_v = self.w_v_latent.weight.view(
            self.num_heads,
            self.head_dim,
            self.latent_dim
        )  # (H,D,C)

        attn = torch.einsum(
            "bhlc,hdc->bhld",
            attn_c,
            w_v
        )  # (B,H,L,D)

        output = attn.transpose(1, 2).contiguous().view(
            B, L, self.total_q_dim
        )

        return self.w_o(output)

# -------------------------- 3. 算力/显存/速度对比函数 --------------------------
def compare_performance(model, x, model_name):
    print(f"\n{'='*50}")
    print(f"对比模型：{model_name}")
    print(f"{'='*50}")
    
    # 1. 参数量统计
    params = sum(p.numel() for p in model.parameters())
    print(f"总参数量：{params/1e6:.2f} M")
    
    # 2. FLOPs计算
    macs, _ = profile(model, inputs=(x,), verbose=False)
    flops = 2 * macs  # MACs乘2得到FLOPs
    print(f"前向FLOPs：{flops/1e9:.2f} G")
    
    # 3. 显存占用统计（CUDA环境）
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        _ = model(x)
        torch.cuda.synchronize()
        peak_memory = torch.cuda.max_memory_allocated()
        print(f"峰值显存占用：{peak_memory/1e6:.2f} MB")
    
    # 4. 运行速度统计
    warmup_runs = 10
    test_runs = 50
    for _ in range(warmup_runs):
        _ = model(x)
    torch.cuda.synchronize()
    
    start_time = time.time()
    for _ in range(test_runs):
        _ = model(x)
    torch.cuda.synchronize()
    avg_time = (time.time() - start_time) / test_runs * 1000
    print(f"平均前向时间：{avg_time:.2f} ms")
    
    return params, flops, peak_memory, avg_time

# -------------------------- 4. 运行对比 --------------------------
if __name__ == "__main__":
    # 生成输入
    x = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM).to(DEVICE)
    
    # 初始化模型
    mha = StandardMHA(HIDDEN_DIM, NUM_HEADS, HEAD_DIM).to(DEVICE)
    mla = MultiHeadLatentAttention(HIDDEN_DIM, NUM_HEADS, HEAD_DIM, LATENT_DIM).to(DEVICE)
    
    # 验证数学等价性（输出形状一致）
    with torch.no_grad():
        mha_out = mha(x)
        mla_out = mla(x)
        print(f"输入形状：{x.shape}")
        print(f"MHA输出形状：{mha_out.shape}")
        print(f"MLA输出形状：{mla_out.shape}")
        assert mha_out.shape == mla_out.shape, "输出形状不一致！"
    
    # 性能对比
    mha_stats = compare_performance(mha, x, "标准MHA")
    mla_stats = compare_performance(mla, x, "MLA")
    
    # 计算节省比例
    print(f"\n{'='*50}")
    print("MLA 相对于 MHA 的节省比例")
    print(f"{'='*50}")
    print(f"参数量节省：{(1 - mla_stats[0]/mha_stats[0])*100:.1f}%")
    print(f"FLOPs节省：{(1 - mla_stats[1]/mha_stats[1])*100:.1f}%")
    if DEVICE == "cuda":
        print(f"显存节省：{(1 - mla_stats[2]/mha_stats[2])*100:.1f}%")
    print(f"速度提升：{(mha_stats[3]/mla_stats[3] - 1)*100:.1f}%")

输入形状：torch.Size([2, 2048, 4096])
MHA输出形状：torch.Size([2, 2048, 4096])
MLA输出形状：torch.Size([2, 2048, 4096])

对比模型：标准MHA
总参数量：67.11 M
前向FLOPs：549.76 G
峰值显存占用：3917.48 MB
平均前向时间：15.60 ms

对比模型：MLA
总参数量：39.85 M
前向FLOPs：292.06 G
峰值显存占用：5268.05 MB
平均前向时间：20.04 ms

MLA 相对于 MHA 的节省比例
参数量节省：40.6%
FLOPs节省：46.9%
显存节省：-34.5%
速度提升：-22.2%
